# Friends Demo — Phase 5: ADD / UPDATE / DELETE / NOOP

The claim: Mem0 compares a new fact against similar existing memories and decides to **add**,
**update**, **delete**, or **ignore (NOOP)** it. Phase 2 tested one case of this incidentally
(Maya's pottery-to-painting switch) and found `add()` only ever reported ADD. This notebook
tests all four branches on purpose, one at a time, so we get real evidence for each claim
instead of one anecdote.

Same Maya/Jordan/Sam data, continued from earlier notebooks.


## 0. Setup and starting snapshot

In [1]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

starting = client.get_all(filters={"user_id": "sam"})
starting_facts = [item["memory"] for item in starting.get("results", [])]

print(f"Starting count for Sam: {len(starting_facts)}")
for f in starting_facts:
    print(" -", f)

Starting count for Sam: 3
 - Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time.
 - User is allergic to peanuts and requests no peanut sauce in future meals
 - User got really into a new video game during the week of August 5, 2026


## Test 1 — ADD

A fact unrelated to anything already stored about Sam -- nothing for mem0 to compare it
against.


In [4]:
before = client.get_all(filters={"user_id": "sam"})
before_count = len(before.get("results", []))

add_result = client.add(
    "Sam just adopted a cat named Pixel.",
    user_id="sam",
)
print("add() returned:", add_result)

after = client.get_all(filters={"user_id": "sam"})
after_count = len(after.get("results", []))

print(f"\nCount before: {before_count}, after: {after_count}")
print("ADD confirmed" if after_count > before_count else "No new memory was actually stored")

add() returned: {'event_id': '8b7c076d-41e9-4bf8-a5bb-9411919474b3', 'status': 'PENDING'}

Count before: 4, after: 4
No new memory was actually stored


## Test 2 — UPDATE

A correction to something already stored -- Sam's peanut allergy detail, corrected.


In [6]:
before = client.get_all(filters={"user_id": "sam"})
before_facts = {item["memory"] for item in before.get("results", [])}

update_result = client.add(
    "Correction: I'm actually allergic to tree nuts, not peanuts -- peanuts are fine, "
    "I misspoke earlier.",
    user_id="sam",
)
print("add() returned:", update_result)

after = client.get_all(filters={"user_id": "sam"})
after_facts = {item["memory"] for item in after.get("results", [])}

removed = before_facts - after_facts
added = after_facts - before_facts

print("\nFacts removed (should be the old, now-wrong allergy info, if UPDATE edits in place):")
for f in removed:
    print(" -", f)

print("\nFacts added:")
for f in added:
    print(" -", f)

print(f"\nCount before: {len(before_facts)}, after: {len(after_facts)}")
print("Looks like an in-place UPDATE" if removed and added and len(after_facts) == len(before_facts)
      else "Did NOT behave like a clean in-place update -- see the raw before/after above")

add() returned: {'event_id': 'a04d0bf8-10c2-4981-bfa8-695de869d104', 'status': 'PENDING'}

Facts removed (should be the old, now-wrong allergy info, if UPDATE edits in place):

Facts added:

Count before: 5, after: 5
Did NOT behave like a clean in-place update -- see the raw before/after above


## Test 3 — DELETE

A direct contradiction, isolated on its own. Jordan claims to have stopped playing piano
entirely.


In [8]:
before = client.get_all(filters={"user_id": "jordan"})
before_facts = {item["memory"] for item in before.get("results", [])}
before_count = len(before_facts)

delete_result = client.add(
    "Correction: I actually stopped playing piano months ago -- I don't play anymore at all, "
    "that's out of date.",
    user_id="jordan",
)
print("add() returned:", delete_result)

after = client.get_all(filters={"user_id": "jordan"})
after_facts = {item["memory"] for item in after.get("results", [])}
after_count = len(after_facts)

removed = before_facts - after_facts

print(f"\nCount before: {before_count}, after: {after_count}")
print("\nFacts actually removed:")
if removed:
    for f in removed:
        print(" -", f)
else:
    print(" (none -- the old piano fact is still present)")

print("\nDELETE confirmed" if removed else "DELETE did NOT happen")

add() returned: {'event_id': '8a73e765-6939-4513-92b3-e2e83ee7e234', 'status': 'PENDING'}

Count before: 7, after: 7

Facts actually removed:
 (none -- the old piano fact is still present)
DELETE did NOT happen


## Test 4 — NOOP

Restating something already known, in different words. Count should stay the same.


In [9]:
before = client.get_all(filters={"user_id": "jordan"})
before_count = len(before.get("results", []))

noop_result = client.add(
    "Just to say it again, I really enjoy cooking big meals for friends.",
    user_id="jordan",
)
print("add() returned:", noop_result)

after = client.get_all(filters={"user_id": "jordan"})
after_count = len(after.get("results", []))

print(f"\nCount before: {before_count}, after: {after_count}")
print("NOOP confirmed -- count unchanged" if after_count == before_count
      else f"Count changed by {after_count - before_count} -- NOOP did not hold")

add() returned: {'event_id': 'fd296466-7930-4d5e-8bd1-cd7b237dc2dc', 'status': 'PENDING'}

Count before: 7, after: 7
NOOP confirmed -- count unchanged


## Summary table

Fill this in from your actual output above -- don't assume the slide's description before
checking.

| Case | Claimed behavior | What we actually observed |
|---|---|---|
| ADD | New, unrelated fact gets stored | *(fill in from Test 1)* |
| UPDATE | Correction edits the existing memory in place | *(fill in from Test 2)* |
| DELETE | Direct contradiction removes the old fact | *(fill in from Test 3)* |
| NOOP | Restating known info changes nothing | *(fill in from Test 4)* |

If ADD and NOOP hold up but UPDATE and DELETE don't behave as cleanly as claimed, that
matches the same finding from the aircraft example: **the write path leans on ADD far more
than the architecture description suggests, and cleanup is mostly left to the read path
(ranking, decay) instead.** Two different examples landing on the same conclusion is a
stronger claim for the demo than either one alone.
